In [ ]:
!pip install sentence-transformers faiss-cpu

In [1]:
import os
import re
import getpass
import requests
import openai
from openai import OpenAI
import jsonschema
import os
import json
from typing import List, Optional, Dict, Any

import pandas as pd
from docx import Document
from docx.shared import Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
from pydantic import BaseModel, Field
from dotenv import load_dotenv

In [19]:
load_dotenv()

# ==== НАСТРОЙКИ ====
OPENAI_API_KEY = getpass.getpass("Введи ваш VseGPT ключ API:")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["OPENAI_BASE_URL"] = "https://api.vsegpt.ru/v1"
OPENAI_MODEL = "openai/gpt-4o-mini"

# Пути к файлам
CSV_CRITERIA_PATH = r"C:\Users\troyd\OneDrive\Desktop\Стажировка\Критерии.csv"  # файл во вложении
DOCX_SPEC_PATH = r"C:\Users\troyd\OneDrive\Desktop\Стажировка\1_Приложение №1_ТЗ_сейсмика.docx"
OUTPUT_REPORT_DOCX = r"C:\Users\troyd\OneDrive\Desktop\Стажировка\Analysis_Report_SGR.docx"

if not OPENAI_API_KEY or OPENAI_API_KEY == "YOUR_OPENAI_API_KEY_HERE":
    raise ValueError("Не задан OPENAI_API_KEY. Установите переменную окружения или впишите ключ.")

print("✅ Конфигурация загружена")

✅ Конфигурация загружена


In [3]:
class GeneratedCriteria(BaseModel):
    selected_ids: List[int] = Field(..., description="ID критериев из справочника, которые релевантны ТЗ")
    new_criteria: List[Dict[str, Any]] = Field(..., description="Список новых критериев: [{'name': '...', 'description': '...', 'importance': int}]")

class CriterionReasoning(BaseModel):
    """Пошаговое рассуждение для одного критерия (Schema-Guided Reasoning)."""
    
    criterion_id: int = Field(..., description="Порядковый номер критерия")
    criterion_name: str = Field(..., description="Название критерия")
    criterion_description: str = Field(..., description="Описание из CSV")
    
    importance_level: int = Field(..., description="Определи уровень важности этого критерия для данного ТЗ")

    criterion_understanding: str = Field(
        ...,
        description="Объясни, ЧТО проверяет этот критерий (1–2 предложения)."
    )

    relevant_sections: str = Field(
        ...,
        description="Какие разделы/приложения ТЗ релевантны для этого критерия?"
    )

    reasoning_steps: str = Field(
        ...,
        description="Пошагово объясни: Критерий требует X → В документе найдено Y → Вывод Z."
    )

    status: str = Field(
        ...,
        description="Статус выполнения критерия.",
        json_schema_extra={"enum": ["✅ Выполнено", "⚠️ Частично", "❌ Не выполнено"]} # Исправили warning заодно
    )

    quote_or_evidence: str = Field(
        ...,
        description="Прямая цитата из ТЗ или обоснование статуса."
    )
    
    recommendation: Optional[str] = Field(
        default=None,
        description="Рекомендация по улучшению. Если нет — передать null."
    )


class SpecificationAnalysisWithReasoning(BaseModel):
    """Анализ ТЗ с явным SGR для каждого критерия."""
    
    reasoning_schema_used: bool = Field(
        ..., # Убрали default=True, модель должна сама решить или вы жестко задаете это в промпте
        description="Флаг: используется ли Schema-Guided Reasoning."
    )

    overall_summary: str = Field(
        ...,
        description="Общая оценка качества ТЗ (1-2 абзаца)."
    )

    criteria_analysis: List[CriterionReasoning] = Field(
        ...,
        description="Список проверок ТОЛЬКО по критериям, отобранным как релевантные для данного ТЗ."
    )

    additional_criteria: List[CriterionReasoning] = Field(
        default=[],  # Может быть пустым списком, если модель не нашла дополнительных
        description="Дополнительные критерии, выявленные моделью при анализе ТЗ (проблемы, не покрытые базовым списком)."
    )
    
    # Лучше временно убрать Dict[str, Any] или заменить на конкретную модель, 
    # так как 'Any' плохо работает со строгим режимом.
    # Если метрики не критичны, можно закомментировать поле metrics:
    # metrics: Dict[str, str] = Field(..., description="Метрики анализа (ключ-значение).")


In [4]:
def load_criteria_from_csv(csv_path: str) -> List[Dict[str, Any]]:
    """
    Читает критерии из CSV с разделителем ';' (точка с запятой).
    Ожидаемые колонки: Название, Описание, Важность
    """
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV с критериями не найден: {csv_path}")
    
    # Важно: использовать sep=';' и encoding зависит от системы
    df = pd.read_csv(csv_path, sep=';', encoding='utf-8')
    
    print(f"Загруженные колонки: {df.columns.tolist()}")
    print(f"Первая строка: {df.iloc[0].to_dict()}")
    
    # Проверка обязательных колонок
    required_cols = ["Название", "Описание", "Важность"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"В CSV отсутствует обязательная колонка '{col}'. Колонки: {df.columns.tolist()}")
    
    criteria = []
    for idx, row in df.iterrows():
        criteria.append({
            "id": idx + 1,  # Порядковый номер
            "name": str(row["Название"]).strip(),
            "description": str(row["Описание"]).strip(),
            "importance": int(row["Важность"]),
        })
    
    print(f"✅ Загружено критериев: {len(criteria)}")
    return criteria


def extract_text_from_docx(docx_path: str) -> str:
    """
    Извлекает текст из DOCX файла.
    """
    if not os.path.exists(docx_path):
        raise FileNotFoundError(f"DOCX файл не найден: {docx_path}")
    
    doc = Document(docx_path)
    paragraphs = [p.text for p in doc.paragraphs if p.text.strip()]
    full_text = "\n".join(paragraphs)
    
    print(f"✅ Текст ТЗ извлечён ({len(full_text)} символов)")
    return full_text


In [17]:
def create_criteria_selection_prompt(spec_text: str, all_criteria: List[Dict[str, Any]]) -> str:
    """
    ЭТАП 1: Анализ ТЗ и формирование списка проверок.
    """
    criteria_list_str = "\n".join([
        f"{c['id']}. {c['name']} (Важность: {c['importance']})\n {c['description']}"
        for c in all_criteria
    ])
    
    return f"""
Ты — методолог по системному анализу. Твоя задача — изучить ТЗ и составить достаточный список критериев для проверки.

СТРОГИЕ ПРАВИЛА:
1. Используй ТОЛЬКО информацию из текста ТЗ. Запрещено опираться на внешние знания, стандарты или практики, не упомянутые в ТЗ.
2. Если в ТЗ нет информации по какому-либо аспекту — НЕ генерируй критерий для его проверки.
3. Максимум 3 новых критерия. Генерируй их только если они:
   - Явно вытекают из текста ТЗ,
   - Критически важны (важность 4–5),
   - Проверяемы по тексту ТЗ без домыслов.

Алгоритм работы:
1. Проанализируй ТЗ: определи тип системы и предметную область.
2. Выбери из Справочника ID тех критериев, которые применимы к этому ТЗ.
3. Если есть критические аспекты, не покрытые Справочником и явно присутствующие в ТЗ — добавь не более 3 новых критериев.

ЗАПРЕЩЕНО:
- Генерировать критерии для аспектов, не упомянутых в ТЗ,
- Предполагать наличие модулей, ролей, интеграций, не описанных в ТЗ.

СПРАВОЧНИК КРИТЕРИЕВ:
{criteria_list_str}

ТЕХНИЧЕСКОЕ ЗАДАНИЕ:
{spec_text[:8000]}

Верни JSON с полями:
- selected_ids: массив ID из справочника,
- generated_criteria: массив новых критериев (максимум 3, можно пустой массив).
"""


def create_analysis_prompt(spec_text: str, final_criteria: List[Dict[str, Any]]) -> str:
    """
    ЭТАП 2: Детальный анализ по утвержденному списку.
    Промпт с обязательным цитированием и запретом домыслов.
    """
    criteria_list = "\n".join([
        f"ID {c['id']}: {c['name']}\nОписание: {c['description']}"
        for c in final_criteria
    ])

    return f"""
Ты — системный аудитор технической документации. Проведи аудит ТЗ по СТРОГО ЗАДАННОМУ списку критериев.

КРИТИЧЕСКИ ВАЖНЫЕ ПРАВИЛА:
1. ОБЯЗАТЕЛЬНОЕ ЦИТИРОВАНИЕ: в поле `quote_or_evidence` ВСЕГДА вставляй дословную цитату из ТЗ в кавычках «...».
2. Если информация по критерию НЕ НАЙДЕНА в ТЗ:
   - status: «❌ Не выполнено»,
   - quote_or_evidence: «В тексте ТЗ отсутствует информация по этому критерию.»,
   - reasoning_steps: опиши, что именно искал, где искал и почему считаешь, что информации нет.
3. ЗАПРЕЩЕНО использовать внешние знания, стандарты, предположения.
4. ЗАПРЕЩЕНО писать «вероятно», «обычно», «можно предположить» и подобные формулировки.

Инструкция для КАЖДОГО критерия (Schema-Guided Reasoning):

1. criterion_understanding:
   Кратко (1–2 предложения), что именно проверяет критерий, строго опираясь на его описание.

2. relevant_sections:
   Какие разделы/пункты ТЗ ты проверял (например, названия разделов или «Весь документ»).

3. reasoning_steps:
   Пошаговая логика:
   - Шаг 1: Критерий требует наличия/описания X.
   - Шаг 2: В разделе Y найдено / НЕ найдено Z.
   - Шаг 3: Вывод: выполнено / частично / не выполнено.

4. status:
   - ✅ Выполнено: информация есть в достаточном объёме,
   - ⚠️ Частично: информация есть, но неполная/неточная,
   - ❌ Не выполнено: информации нет или её недостаточно.

5. quote_or_evidence:
   ОБЯЗАТЕЛЬНО одно из двух:
   А) Дословная цитата из ТЗ: «текст из документа»;
   Б) «В тексте ТЗ отсутствует информация по этому критерию.»

6. recommendation:
   - Если статус ❌ или ⚠️: конкретная рекомендация, что добавить/уточнить в ТЗ.
   - Если статус ✅: передай null.

Для критериев, имя которых начинается с «[AI]», будь максимально консервативен: если нет явной поддержки в ТЗ, ставь статус «❌ Не выполнено».

СПИСОК КРИТЕРИЕВ ДЛЯ ПРОВЕРКИ:
{criteria_list}

ТЕХНИЧЕСКОЕ ЗАДАНИЕ (ПОЛНЫЙ ТЕКСТ):
{spec_text}

Проверь ВСЕ критерии из списка. Не пропускай ни одного.
"""



In [15]:
class SpecAnalyzerWithSGR:
    """Анализатор ТЗ с двухэтапным процессом: Генерация критериев -> SGR Анализ."""
    
    def __init__(self, api_key: str, model: str = "openai/gpt-4o-mini"):
        self.client = OpenAI(
            api_key=api_key,
            base_url="https://api.vsegpt.ru/v1"
        )
        self.model = model
        print(f"✅ Инициализирован анализатор VseGPT ({model})")

    def select_and_generate_criteria(self, spec_text: str, all_criteria: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """ЭТАП 1: Выбирает существующие и генерирует новые критерии."""
        
        prompt = create_criteria_selection_prompt(spec_text, all_criteria)
        
        # Схема для этапа генерации
        generation_schema = {
            "name": "CriteriaSelection",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "selected_ids": {
                        "type": "array",
                        "items": {"type": "integer"},
                        "description": "Список ID критериев из справочника, которые строго применимы к этому ТЗ."
                    },
                    "generated_criteria": {
                        "type": "array",
                        "maxItems": 3,
                        "items": {
                            "type": "object",
                            "properties": {
                                "name": {
                                    "type": "string",
                                    "maxLength": 450
                                },
                                "description": {
                                    "type": "string",
                                    "maxLength": 1500
                                },
                                "importance": {
                                    "type": "integer",
                                    "minimum": 4,
                                    "maximum": 5,
                                    "description": "Важность 4–5 (генерируем только критичные)."
                                }
                            },
                            "required": ["name", "description", "importance"],
                            "additionalProperties": False
                        },
                        "description": "Список НОВЫХ критериев (макс. 3), только если они критичны и проверяемы по ТЗ."
                    }
                },
                "required": ["selected_ids", "generated_criteria"],
                "additionalProperties": False
            }
        }


        print(f"🕵️ ЭТАП 1: Формирование списка критериев...")
        try:
            response = self.client.chat.completions.create(
            model=self.model,
            temperature=0.2,
            top_p=0.85,
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_schema", "json_schema": generation_schema}
            )
            
            data = json.loads(response.choices[0].message.content)
            
            # Сборка финального списка
            final_list = []
            
            # 1. Добавляем выбранные из CSV
            selected_ids = set(data.get('selected_ids', []))
            for crit in all_criteria:
                if crit['id'] in selected_ids:
                    final_list.append(crit)
            
            # 2. Добавляем сгенерированные (присваиваем новые ID)
            # Начинаем нумерацию после последнего ID из CSV
            max_id = max([c['id'] for c in all_criteria], default=0)
            current_id = max_id + 1
            
            for new_crit in data.get('generated_criteria', []):
                new_crit_obj = {
                    "id": current_id,
                    "name": f"[AI] {new_crit['name']}", # Пометка, что критерий от ИИ
                    "description": new_crit['description'],
                    "importance": new_crit['importance']
                }
                final_list.append(new_crit_obj)
                current_id += 1
                
            print(f"   ✅ Выбрано из базы: {len(selected_ids)}")
            print(f"   ✨ Сгенерировано новых: {len(data.get('generated_criteria', []))}")
            print(f"   📋 Итого к проверке: {len(final_list)}")
            
            return final_list

        except Exception as e:
            print(f"❌ Ошибка на этапе генерации: {e}")
            raise

    def analyze_with_sgr(self, spec_text: str, criteria: List[Dict[str, Any]]) -> SpecificationAnalysisWithReasoning:
        """ЭТАП 2: Анализ по готовому списку."""
        
        prompt = create_analysis_prompt(spec_text, criteria)

        # ИСПРАВЛЕНИЕ: Полностью убрали additional_criteria из схемы,
        # так как на этом этапе мы их не генерируем.
        json_schema = {
            "name": "SpecificationAnalysisWithReasoning",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "reasoning_schema_used": {"type": "boolean"},
                    "overall_summary": {
                        "type": "string",
                        "maxLength": 1500
                    },
                    "criteria_analysis": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "criterion_id": {"type": "integer"},
                                "criterion_name": {"type": "string"},
                                "criterion_description": {"type": "string"},
                                "importance_level": {
                                    "type": "integer",
                                    "minimum": 1,
                                    "maximum": 5
                                },
                                "criterion_understanding": {
                                    "type": "string",
                                    "maxLength": 1400
                                },
                                "relevant_sections": {
                                    "type": "string",
                                    "maxLength": 1300
                                },
                                "reasoning_steps": {
                                    "type": "string",
                                    "maxLength": 1800
                                },
                                "status": {
                                    "type": "string",
                                    "enum": ["✅ Выполнено", "⚠️ Частично", "❌ Не выполнено"]
                                },
                                "quote_or_evidence": {
                                    "type": "string",
                                    "minLength": 10,
                                    "description": "Дословная цитата в кавычках или явное указание, что информация отсутствует."
                                },
                                "recommendation": {
                                    "type": ["string", "null"]
                                }
                            },
                            "required": [
                                "criterion_id", "criterion_name", "criterion_description",
                                "importance_level", "criterion_understanding", "relevant_sections",
                                "reasoning_steps", "status", "quote_or_evidence", "recommendation"
                            ],
                            "additionalProperties": False
                        }
                    }
                },
                "required": ["reasoning_schema_used", "overall_summary", "criteria_analysis"],
                "additionalProperties": False
            }
        }


        print(f"🔬 ЭТАП 2: Детальный анализ ({len(criteria)} критериев)...")
        
        response = self.client.chat.completions.create(
            model=self.model,
            max_tokens=8000,
            temperature=0.1,
            top_p=0.8,
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_schema", "json_schema": json_schema}
        )
        
        data = json.loads(response.choices[0].message.content)
        
        # Ручная установка дефолтного значения для совместимости с Pydantic,
        # если оно не пришло от модели
        if "additional_criteria" not in data:
            data["additional_criteria"] = []
        
        return SpecificationAnalysisWithReasoning(**data)



In [10]:
def export_sgr_analysis_to_docx(
    analysis: SpecificationAnalysisWithReasoning,
    source_docx_path: str,
    output_path: str
):
    """
    Экспортирует результат SGR-анализа в DOCX-отчёт.
    """
    doc = Document()

    # Заголовок
    title = doc.add_heading("Отчёт по анализу ТЗ", level=1)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER

    doc.add_paragraph(f"Исходный документ: {os.path.basename(source_docx_path)}")
    doc.add_paragraph(" ")

    # Общая оценка
    doc.add_heading("1. Общая оценка", level=2)
    doc.add_paragraph(analysis.overall_summary)

    # Метрики
    '''if analysis.metrics:
        doc.add_heading("2. Метрики", level=2)
        for k, v in analysis.metrics.items():
            doc.add_paragraph(f"• {k}: {v}")
    doc.add_paragraph(" ")'''

    # Детальный анализ
    doc.add_heading("3. Детальный анализ по критериям", level=2)

    for crit in analysis.criteria_analysis:
        # Заголовок критерия
        heading_text = f"3.{crit.criterion_id} {crit.criterion_name} (Важность: {crit.importance_level})"
        doc.add_heading(heading_text, level=3)

        # Статус (жирный)
        p = doc.add_paragraph()
        run = p.add_run(f"Статус: ")
        run.bold = True
        p.add_run(crit.status)

        # Описание из CSV (серый текст)
        doc.add_paragraph(f"Определение: {crit.criterion_description}", style="List Bullet")

        # Понимание
        '''p = doc.add_paragraph()
        run = p.add_run("Понимание критерия: ")
        run.bold = True
        doc.add_paragraph(crit.criterion_understanding)'''

        # Релевантные разделы
        p = doc.add_paragraph()
        run = p.add_run("Релевантные разделы ТЗ: ")
        run.bold = True
        doc.add_paragraph(crit.relevant_sections)

        # Рассуждение
        '''p = doc.add_paragraph()
        run = p.add_run("Логическое рассуждение: ")
        run.bold = True
        doc.add_paragraph(crit.reasoning_steps)'''

        # Доказательство
        p = doc.add_paragraph()
        run = p.add_run("Доказательство / цитата: ")
        run.bold = True
        doc.add_paragraph(crit.quote_or_evidence)

        # Рекомендация (если есть)
        if crit.recommendation and crit.recommendation.lower() != "нет":
            p = doc.add_paragraph()
            run = p.add_run("Рекомендация: ")
            run.bold = True
            doc.add_paragraph(crit.recommendation)

        
        doc.add_paragraph(" ")  # Разделитель

    # Сохранение
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    doc.save(output_path)
    print(f"✅ Отчёт сохранён: {output_path}")


In [20]:
print("=" * 60)
print("ДВУХЭТАПНЫЙ АНАЛИЗ ТЗ: ГЕНЕРАЦИЯ + ПРОВЕРКА")
print("=" * 60)

# 1. Загружаем критерии из CSV
try:
    base_criteria = load_criteria_from_csv(CSV_CRITERIA_PATH)
except Exception as e:
    print(f"❌ Ошибка при чтении CSV: {e}")
    raise

# 2. Извлекаем текст ТЗ
try:
    spec_text = extract_text_from_docx(DOCX_SPEC_PATH)
except Exception as e:
    print(f"❌ Ошибка при чтении DOCX: {e}")
    raise

# 3. Инициализация
analyzer_sgr = SpecAnalyzerWithSGR(api_key=OPENAI_API_KEY, model=OPENAI_MODEL)

try:
    # --- ШАГ 1: Отбор и Генерация критериев ---
    # Модель читает ТЗ и формирует уникальный список проверок
    final_criteria_pool = analyzer_sgr.select_and_generate_criteria(
        spec_text=spec_text, 
        all_criteria=base_criteria
    )
    
    # --- ШАГ 2: Детальный анализ (SGR) ---
    # Модель проверяет ТЗ по сформированному списку
    analysis_result = analyzer_sgr.analyze_with_sgr(
        spec_text=spec_text,
        criteria=final_criteria_pool
    )

    # 4. Вывод результатов в консоль
    print("\n" + "=" * 60)
    print("РЕЗУЛЬТАТЫ АНАЛИЗА (Топ-5 критериев)")
    print("=" * 60)

    for crit in analysis_result.criteria_analysis[:5]:
        print(f"\n📌 {crit.criterion_name} (ID: {crit.criterion_id})")
        print(f"   Статус: {crit.status}")
        print(f"   Вывод: {crit.quote_or_evidence[:100]}...")

    # 5. Экспорт
    export_sgr_analysis_to_docx(
        analysis=analysis_result,
        source_docx_path=DOCX_SPEC_PATH,
        output_path=OUTPUT_REPORT_DOCX
    )

except Exception as e:
    print(f"\n❌ КРИТИЧЕСКАЯ ОШИБКА ПРОЦЕССА: {e}")
    # Для отладки можно вывести traceback
    import traceback
    traceback.print_exc()

print("\n" + "=" * 60)
print("✅ ГОТОВО! Отчет сформирован.")
print("=" * 60)


ДВУХЭТАПНЫЙ АНАЛИЗ ТЗ: ГЕНЕРАЦИЯ + ПРОВЕРКА
Загруженные колонки: ['Название', 'Описание', 'Важность']
Первая строка: {'Название': 'Полнота функциональных требований', 'Описание': 'Оценка того, насколько подробно описано, что должна делать система, включая объекты, бизнес-логику, роли и отчётность.', 'Важность': 3}
✅ Загружено критериев: 26
✅ Текст ТЗ извлечён (9140 символов)
✅ Инициализирован анализатор VseGPT (openai/gpt-4o-mini)
🕵️ ЭТАП 1: Формирование списка критериев...
   ✅ Выбрано из базы: 6
   ✨ Сгенерировано новых: 3
   📋 Итого к проверке: 9
🔬 ЭТАП 2: Детальный анализ (9 критериев)...

РЕЗУЛЬТАТЫ АНАЛИЗА (Топ-5 критериев)

📌 Полнота функциональных требований (ID: 1)
   Статус: ✅ Выполнено
   Вывод: /null/ (нет необходимости в цитате)...

📌 Бизнес-логика (ID: 3)
   Статус: ✅ Выполнено
   Вывод: /null/ (нет необходимости в цитате)...

📌 Технические и архитектурные требования (ID: 6)
   Статус: ✅ Выполнено
   Вывод: /null/ (нет необходимости в цитате)...

📌 Нефункциональные требов